# # Análise Exploratória de Dados de Vazão de Poço
# **Instrumento:** Medidor de Vazão Ultrassônico Portátil (Clamp-on / Tempo de Trânsito)  
# **Arquivo de Entrada:** `Dados/P1_Str0248.csv`  
# **Contexto de Automação:** Diagnóstico de regime de bombeamento de poço tubular profundo, integridade metrológica dos sensores e balanço volumétrico.

# ## Bloco 1: Importação de Bibliotecas e Configuração do Ambiente
# Importamos as bibliotecas fundamentais para manipulação de dados (`pandas`, `numpy`) e visualização gráfica (`matplotlib`).

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Configurações visuais para gráficos técnicos de engenharia
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.4
plt.rcParams['grid.linestyle'] = '--'

print("Ambiente configurado com sucesso!")
print(f"Versão do Pandas: {pd.__version__}")

# ## Bloco 2: Investigação Forense do Arquivo Bruto
# Antes de carregar os dados, realizamos uma inspeção preliminar no arquivo `P1_Str0248.csv`.
# Em medições de campo com equipamentos portáteis, os arquivos brutos frequentemente trazem peculiaridades de firmware ou cabeçalhos proprietários.

# ### 2.1 O Problema da Leitura Direta (Ingênua) com Pandas
# Se tentarmos ler o arquivo diretamente com `pd.read_csv(caminho_csv)` sem tratamento prévio, o Pandas não apresentará um erro fatal de carregamento, mas corromperá a integridade do dataset de várias formas:
# 1. As 20 colunas de dados (com vírgula residual no final) contra 19 no cabeçalho fazem o Pandas usar a primeira coluna como índice acidentalmente.
# 2. A linha de unidades físicas (`m/s`, `m^3/h`, etc.) é tratada como se fosse dado, forçando **todas** as grandezas numéricas para o tipo texto (`str`).
# 3. Qualquer cálculo matemático (como `df['Volumetric'].mean()`) falha imediatamente com `TypeError`.

In [ ]:
caminho_csv = os.path.join("Dados", "P1_Str0248.csv")

# Leitura ingênua demonstrativa
df_ingenuo = pd.read_csv(caminho_csv)

print(f"Shape do DataFrame ingênuo: {df_ingenuo.shape} (Deveria ter 900 linhas e 19 colunas)")
print("\nTipos de dados detectados pelo Pandas (todas as medidas viraram texto):")
print(df_ingenuo.dtypes.head(6))

print("\nTentativa de calcular a média da vazão volumétrica na leitura ingênua:")
try:
    media_vazao = df_ingenuo["Volumetric"].mean()
except Exception as erro:
    print(f" Falha no cálculo: {type(erro).__name__}: {erro}")

print("\nPrimeiras 2 linhas do DataFrame ingênuo (note a linha de unidades misturada nos dados):")
df_ingenuo.head(2)

# ### 2.2 Inspeção em Modo Texto Puro
# O erro `ParserError: Expected 19 fields in line 2, saw 20` indica uma divergência no número de colunas entre a linha 1 e as linhas seguintes.  
# Vamos abrir o arquivo em modo texto puro com Python nativo e analisar as primeiras e últimas linhas em formato explícito (`repr()`).

In [ ]:
with open(caminho_csv, "r", encoding="utf-8", errors="ignore") as f:
    primeiras_linhas = [f.readline() for _ in range(5)]

print("=== INSPEÇÃO DAS PRIMEIRAS LINHAS (Modo RAW) ===")
for i, linha in enumerate(primeiras_linhas, start=1):
    campos = linha.strip().split(",")
    print(f"\nLinha {i} ({len(campos)} campos):")
    print(f"Conteúdo: {repr(linha)}")

# ### 2.3 Diagnóstico dos Problemas Identificados
# 
# 1. **Caracteres Nulos (`\x00`) na Linha 1:**
#    - O firmware do medidor inseriu o byte nulo `\x00` antes de `Error`, antes de `Sound Speed` e no final de `Signal to Noise Ratio Down\x00`.
#    - Se não for limpo, os nomes das colunas ficam corrompidos na indexação do DataFrame.
# 
# 2. **Divergência de Colunas (19 vs 20):**
#    - A **Linha 1 (Cabeçalho)** tem 19 campos e **não** termina com vírgula.
#    - A **Linha 2 (Unidades)** e as **Linhas 3 em diante (Dados)** terminam com uma **vírgula residual** `,`, gerando 20 campos (onde o último campo é uma string vazia).
#    - É isso que causa o `ParserError` no Pandas.
# 
# 3. **Linha 2 com Unidades de Engenharia:**
#    - A segunda linha não contém dados, mas sim as unidades físicas das grandezas (`m/s`, `m^3/h`, `db`, etc.).
# 
# 4. **Rodapé com Mensagem de Fim de Canal (Linha 903):**
#    - Vamos verificar o final do arquivo:

In [ ]:
with open(caminho_csv, "r", encoding="utf-8", errors="ignore") as f:
    todas_linhas = f.readlines()

print(f"Total de linhas no arquivo: {len(todas_linhas)}")
print(f"Linha 901 (dados) : {repr(todas_linhas[900][:80])}...")
print(f"Linha 902 (dados) : {repr(todas_linhas[901][:80])}...")
print(f"Linha 903 (rodapé): {repr(todas_linhas[902])}")

# Detectamos que a linha 903 traz: 'Congrats! Channel 1 Linear END!\n'
# Esta linha não contém números e causará erro se não for excluída da carga.

# ## Bloco 3: Estratégia de Engenharia de Dados (ETL e Sanitização)
# Com o diagnóstico concluído, criamos um pipeline de carga robusto:
# 1. Sanitizar a linha 1 removendo `\x00` e espaços em branco para obter os nomes limpos de colunas.
# 2. Extrair a linha 2 para mapear as unidades físicas de cada variável.
# 3. Carregar o arquivo com `pd.read_csv()` usando:
#    - `skiprows=2`: ignora as linhas 1 e 2 (cabeçalho e unidades).
#    - `skipfooter=1`: descarta a linha de rodapé `Congrats!...`.
#    - `engine='python'`: habilita suporte completo a `skipfooter`.
#    - `usecols=range(len(colunas))`: lê apenas as 19 colunas úteis, ignorando a vírgula residual no fim.
#    - `names=colunas`: aplica os nomes já sanitizados.
# 4. Converter colunas temporais e criar índice relativo em minutos.

In [ ]:
# 1. Sanitização do cabeçalho e extração de unidades
with open(caminho_csv, "r", encoding="utf-8", errors="ignore") as f:
    linha1_limpa = f.readline().replace("\x00", "").strip()
    linha2_limpa = f.readline().replace("\x00", "").strip()

colunas_limpas = [c.strip() for c in linha1_limpa.split(",") if c.strip()]
unidades_brutas = [u.strip() for u in linha2_limpa.split(",") if u.strip()]

print(f"Total de colunas válidas identificadas: {len(colunas_limpas)}")
print(f"Colunas: {colunas_limpas}")

# 2. Carga estruturada com Pandas
df = pd.read_csv(
    caminho_csv,
    skiprows=2,
    skipfooter=1,
    engine="python",
    header=None,
    usecols=range(len(colunas_limpas)),
    names=colunas_limpas
)

# 3. Limpeza de espaços em branco nos campos textuais
for col in df.select_dtypes(include=["object", "str"]).columns:
    df[col] = df[col].astype(str).str.strip()

# 4. Conversão e sincronização temporal
df["Data_Hora"] = pd.to_datetime(df["Date"] + " " + df["Time"], format="%Y/%m/%d %H:%M:%S")
tempo_zero = df["Data_Hora"].iloc[0]
df["Tempo_Segundos"] = (df["Data_Hora"] - tempo_zero).dt.total_seconds()
df["Tempo_Minutos"] = df["Tempo_Segundos"] / 60.0

print(f"\n Dataset carregado com sucesso: {df.shape[0]} linhas e {df.shape[1]} colunas!")
print(f"Intervalo temporal: {df['Data_Hora'].min()} até {df['Data_Hora'].max()} ({df['Tempo_Minutos'].max():.1f} minutos)")
df.head()

In [ ]:
# Verificação de integridade: tipos de dados e ausência de nulos
print("=== ESTRUTURA DOS DADOS (df.info) ===")
df.info()

print("\n=== VERIFICAÇÃO DE VALORES AUSENTES (NULOS) ===")
print(df.isna().sum())

# ## Bloco 4: Dicionário Técnico das Variáveis e Metadados do Medidor
# Mapeamos as variáveis em duas categorias fundamentais da automação:
# - **Grandezas de Processo / Produção:** Indicam o comportamento hidráulico do poço.
# - **Diagnósticos Acústicos / Metrológicos:** Indicam a saúde física da medição ultrassônica.

In [ ]:
dicionario_metadados = pd.DataFrame({
    "Variável": colunas_limpas,
    "Categoria": [
        "Identificação", "Identificação", "Diagnóstico", "Processo", "Processo", 
        "Processo", "Processo", "Totalização", "Totalização", "Totalização", 
        "Totalização", "Diagnóstico", "Diagnóstico", "Diagnóstico", "Diagnóstico", 
        "Diagnóstico", "Diagnóstico", "Diagnóstico", "Diagnóstico"
    ],
    "Unidade": [
        "-", "-", "Status", "m/s", "m³/h", 
        "Sm³/h", "kg/h", "Litros", "Litros", "Litros", 
        "Segundos", "m/s", "Pontos", "Pontos", "dB", 
        "dB", "dB", "dB", "dB"
    ],
    "Significado Físico / Função no Medidor": [
        "Data do registro da amostragem",
        "Hora da amostragem (taxa de 1 segundo)",
        "Código de erro operacional reportado pelo medidor",
        "Velocidade média do escoamento na seção transversal do tubo",
        "Vazão volumétrica instantânea em condições de operação",
        "Vazão volumétrica padronizada (Standard)",
        "Vazão mássica estimada considerando a densidade da água",
        "Volume acumulado sentido direto (produção útil do poço)",
        "Volume acumulado sentido reverso (refluxo / contra-fluxo)",
        "Volume acumulado líquido (Direto - Reverso)",
        "Tempo acumulado da batelada de medição ativa",
        "Velocidade de propagação do som no fluido (diagnóstico de meio)",
        "Amplitude discreta do sinal acústico a montante (Upstream)",
        "Amplitude discreta do sinal acústico a jusante (Downstream)",
        "Ganho do amplificador eletrônico do canal a montante",
        "Ganho do amplificador eletrônico do canal a jusante",
        "Faixa dinâmica de recepção do sinal acústico",
        "Relação Sinal-Ruído acústico a montante (SNR Up)",
        "Relação Sinal-Ruído acústico a jusante (SNR Down)"
    ]
})

dicionario_metadados

# ## Bloco 5: Análise Estatística Descritiva e Indicadores Operacionais (KPIs)
# Calculamos as estatísticas de dispersão (média, desvio padrão, mínimo, mediana, máximo e coeficiente de variação) e os KPIs de produção hídrica do poço.

In [ ]:
variaveis_processo = ["Velocity", "Volumetric", "Standard Volumetric", "Mass", "Sound Speed"]
estatisticas = df[variaveis_processo].describe().T
estatisticas["amplitude"] = estatisticas["max"] - estatisticas["min"]
estatisticas["cv_%"] = (estatisticas["std"] / estatisticas["mean"]) * 100

print("=== ESTATÍSTICA DESCRITIVA DAS VARIÁVEIS DE PROCESSO ===")
estatisticas[["mean", "std", "min", "50%", "max", "amplitude", "cv_%"]]

In [ ]:
# Cálculo dos KPIs de Produção e Balanço de Vazão
duracao_minutos = df["Tempo_Minutos"].max()
duracao_segundos = len(df)
vazao_media_m3h = df["Volumetric"].mean()
vazao_media_ls = (vazao_media_m3h * 1000.0) / 3600.0
velocidade_media = df["Velocity"].mean()

# Variação do Totalizador do Medidor (Batch Forward Totalizer)
vol_inicial_l = df["Batch Forward Totalizer"].iloc[0]
vol_final_l = df["Batch Forward Totalizer"].iloc[-1]
volume_total_l = vol_final_l - vol_inicial_l
volume_total_m3 = volume_total_l / 1000.0

# Validação cruzada: Integração numérica da vazão média no tempo (Volume = Vazão * Delta_t)
volume_teorico_m3 = vazao_media_m3h * (duracao_minutos / 60.0)
erro_fechamento_balanco = abs(volume_total_m3 - volume_teorico_m3)

print("=" * 65)
print("             INDICADORES CHAVE DE OPERAÇÃO DO POÇO (KPIs)")
print("=" * 65)
print(f"Duração do Ensaio              : {duracao_minutos:.2f} minutos ({duracao_segundos} amostras a 1 Hz)")
print(f"Vazão Média do Poço            : {vazao_media_m3h:.2f} m³/h  ({vazao_media_ls:.2f} L/s)")
print(f"Faixa Operacional de Vazão     : {df['Volumetric'].min():.2f} m³/h a {df['Volumetric'].max():.2f} m³/h")
print(f"Oscilação Máxima (Amplitude)   : {df['Volumetric'].max() - df['Volumetric'].min():.2f} m³/h")
print(f"Estabilidade da Bomba (CV%)    : {estatisticas.loc['Volumetric', 'cv_%']:.2f}% (Regime ultra-estável)")
print(f"Velocidade Média no Tubo       : {velocidade_media:.2f} m/s")
print(f"Volume Total Produzido         : {volume_total_l:,.2f} Litros ({volume_total_m3:.2f} m³)")
print(f"Volume por Integração Teórica  : {volume_teorico_m3:.2f} m³")
print(f"Diferença Metrológica          : {erro_fechamento_balanco:.6f} m³ (Consistência de 100%)")
print("=" * 65)

# ## Bloco 6: Diagnóstico Metrológico Acústico do Medidor
# Em instrumentação ultrassônica clamp-on, a confiabilidade da vazão depende diretamente da integridade do sinal acústico:
# 1. **Coluna `Error`:** Indica se o medidor perdeu rastreio de onda, entrou em timeout ou detectou bolhas.
# 2. **Velocidade do Som no Fluido (`Sound Speed`):**
#    - Água a 25 °C ~ 27 °C possui velocidade acústica teórica entre 1497 m/s e 1503 m/s.
#    - Se houvesse presença de bolhas de ar ou cavitação na sucção da bomba, a velocidade acústica sofreria quedas bruscas ou instabilidade.
# 3. **Relação Sinal-Ruído (`SNR Up` e `SNR Down`):**
#    - Valores acima de 30 dB são o padrão de excelência em sensores clamp-on com bom acoplamento acústico (pasta de acoplamento íntegra).

In [ ]:
status_erro = df["Error"].value_counts()
print("=== STATUS DE ERRO DO MEDIDOR ===")
print(status_erro)

snr_up_medio = df["Signal to Noise Ratio Up"].mean()
snr_down_medio = df["Signal to Noise Ratio Down"].mean()
sound_speed_medio = df["Sound Speed"].mean()
sound_speed_std = df["Sound Speed"].std()

print("\n=== PARÂMETROS ACÚSTICOS DE DIAGNÓSTICO ===")
print(f"Velocidade do Som Média        : {sound_speed_medio:.2f} m/s (Desvio: {sound_speed_std:.4f} m/s)")
print(f"SNR Upstream Médio             : {snr_up_medio:.2f} dB (Mínimo: {df['Signal to Noise Ratio Up'].min():.2f} dB)")
print(f"SNR Downstream Médio           : {snr_down_medio:.2f} dB (Mínimo: {df['Signal to Noise Ratio Down'].min():.2f} dB)")
print(f"Ganho Eletrônico (Up / Down)   : {df['Gain Up'].iloc[0]:.2f} dB / {df['Gain Down'].iloc[0]:.2f} dB (Fixo)")

# Validação automática da qualidade metrológica
aprovado = (
    (status_erro.index[0] == "No Error") and 
    (snr_up_medio >= 30.0) and 
    (snr_down_medio >= 30.0) and 
    (sound_speed_std < 0.2)
)

if aprovado:
    print("\n PARECER DO DIAGNÓSTICO: SINAL EXCELENTE!")
    print("O medidor operou com excelente acoplamento acústico, sem cavitação e sem presença de ar no fluido.")
else:
    print("\n PARECER DO DIAGNÓSTICO: ATENÇÃO NECESSÁRIA. Verificar alinhamento dos sensores ou presença de gás.")

# ## Bloco 7: Visualizações Gráficas Exploratórias
# Geramos os gráficos de séries temporais para análise de processo e diagnóstico.

In [ ]:
# Gráfico 1: Vazão Volumétrica Instantânea vs Tempo (Estabilidade do Poço)
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(df["Tempo_Minutos"], df["Volumetric"], color="#005b96", linewidth=1.5, label="Vazão Instantânea (m³/h)")
ax.axhline(vazao_media_m3h, color="#d9534f", linestyle="--", linewidth=1.8, label=f"Média: {vazao_media_m3h:.2f} m³/h")
ax.fill_between(df["Tempo_Minutos"], df["Volumetric"].min(), df["Volumetric"].max(), color="#005b96", alpha=0.08)

ax.set_title("Vazão Volumétrica do Poço ao Longo do Tempo", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Tempo de Ensaio (minutos)", fontsize=11)
ax.set_ylabel("Vazão Volumétrica (m³/h)", fontsize=11)
ax.legend(loc="upper right", frameon=True)
ax.set_ylim(df["Volumetric"].min() - 0.4, df["Volumetric"].max() + 0.4)

plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 2: Curva de Volume Acumulado do Poço (Totalizador em m³)
fig, ax = plt.subplots(figsize=(12, 5))

volume_acumulado_m3 = (df["Batch Forward Totalizer"] - vol_inicial_l) / 1000.0

ax.plot(df["Tempo_Minutos"], volume_acumulado_m3, color="#2e7d32", linewidth=2.0, label="Volume Acumulado (m³)")
ax.set_title("Volume Total Extraído do Poço (Totalizador Direto)", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Tempo de Ensaio (minutos)", fontsize=11)
ax.set_ylabel("Volume Extraído (m³)", fontsize=11)
ax.legend(loc="upper left", frameon=True)

# Anotação com destaque no ponto de término da medição
ax.annotate(
    f"Total: {volume_total_m3:.2f} m³ ({volume_total_l:,.0f} L)",
    xy=(df["Tempo_Minutos"].iloc[-1], volume_acumulado_m3.iloc[-1]),
    xytext=(df["Tempo_Minutos"].iloc[-1] - 3.8, volume_acumulado_m3.iloc[-1] - 1.2),
    arrowprops=dict(facecolor="#2e7d32", shrink=0.08, width=1.5, headwidth=7),
    fontweight="bold",
    bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="#2e7d32", lw=1.5)
)

plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 3: Diagnóstico Acústico (SNR e Velocidade do Som no Fluido)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# Subgráfico 1: Relação Sinal-Ruído
ax1.plot(df["Tempo_Minutos"], df["Signal to Noise Ratio Up"], label="SNR Upstream (dB)", color="#f0ad4e", linewidth=1.2)
ax1.plot(df["Tempo_Minutos"], df["Signal to Noise Ratio Down"], label="SNR Downstream (dB)", color="#5cb85c", linewidth=1.2)
ax1.axhline(30, color="red", linestyle=":", label="Limite Recomendado de Confiabilidade (30 dB)")
ax1.set_ylabel("SNR (dB)", fontsize=11)
ax1.set_title("Diagnóstico Acústico: SNR e Velocidade do Som no Fluido", fontsize=13, fontweight="bold", pad=12)
ax1.legend(loc="lower right", frameon=True)

# Subgráfico 2: Velocidade do Som
ax2.plot(df["Tempo_Minutos"], df["Sound Speed"], label="Velocidade do Som Medida (m/s)", color="#6f42c1", linewidth=1.5)
ax2.axhline(sound_speed_medio, color="black", linestyle="--", alpha=0.6, label=f"Média: {sound_speed_medio:.2f} m/s")
ax2.set_xlabel("Tempo de Ensaio (minutos)", fontsize=11)
ax2.set_ylabel("Velocidade do Som (m/s)", fontsize=11)
ax2.set_ylim(sound_speed_medio - 0.2, sound_speed_medio + 0.2)
ax2.legend(loc="upper right", frameon=True)

plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 4: Distribuição e Histograma da Vazão Volumétrica
fig, ax = plt.subplots(figsize=(10, 4.5))

n, bins, patches = ax.hist(df["Volumetric"], bins=25, color="#005b96", edgecolor="white", alpha=0.8, density=True)
ax.axvline(vazao_media_m3h, color="#d9534f", linestyle="--", linewidth=2.0, label=f"Média: {vazao_media_m3h:.2f} m³/h")

# Linha de densidade normal teórica
sigma = df["Volumetric"].std()
mu = vazao_media_m3h
x_teorico = np.linspace(df["Volumetric"].min(), df["Volumetric"].max(), 200)
pdf_teorica = (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x_teorico - mu) / sigma)**2)
ax.plot(x_teorico, pdf_teorica, color="#d9534f", linewidth=1.5, label="Ajuste Gaussiano Normal")

ax.set_title("Histograma e Distribuição da Vazão do Poço", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Vazão Volumétrica (m³/h)", fontsize=11)
ax.set_ylabel("Densidade de Probabilidade", fontsize=11)
ax.legend(loc="upper right", frameon=True)

plt.tight_layout()
plt.show()

# ## Bloco 8: Parecer Técnico Consolidado para Automação
# 
# ### 1. Comportamento Operacional do Poço
# - **Regime Estacionário:** A vazão média permaneceu em **46,56 m³/h** (~12,93 L/s), com coeficiente de variação de apenas **0,18%**.  
# - **Estabilidade Hidrogeológica:** A inexistência de oscilações ou tendência de queda durante os 15 minutos comprova que a vazão do poço é superior ou igual à capacidade de extração da bomba no ponto de operação testado (sem esgotamento do nível dinâmico).
# 
# ### 2. Confiabilidade dos Dados do Medidor
# - **Ausência de Erros:** 100% dos 900 registros apresentaram status `No Error`.
# - **Acoplamento Acústico Excelente:** SNR de **36,93 dB** (Up) e **37,18 dB** (Down), com ganho eletrônico estável em **16,05 dB**.
# - **Fluido Homogêneo:** Velocidade do som constante em **1501,88 m/s** com desvio ínfimo (0,02 m/s), comprovando ausência de ar arrastado e temperatura estável da água.
# 
# ### 3. Capacidade de Produção Estimada
# - **Volume em 15 minutos:** **11,63 m³** (11.626 litros).
# - **Projeção Contínua 24h:** Capacidade de bombeamento estimada em **1.117,3 m³/dia** (~1.117.300 L/dia) sob as mesmas condições hidráulicas.